# Final Pipeline — ATE + ASC Inference & Statistics Report
**Dataset:** Electronics Part 1

**Input:** `cleaned_part1_step2.parquet` (output của Phase 2)

**Pipeline:**
1. **Gate Classifier** → lọc câu có aspect (`gate_threshold = 0.7`)
2. **ATE Model** → trích xuất aspect span (`span_threshold = 0.5`)
3. **ASC Model** → phân loại sentiment per aspect (`argmax only`, không dùng threshold)

**Models sử dụng (tốt nhất):**
- Gate: `absa_self_train_phase2/ate_gate_phase2` (Phase 2)
- ATE:  `absa_self_train_phase2/ate_phase2` (Phase 2)
- ASC:  `ASC_PHASE_3/model` (Phase 3 — vòng cuối)

**Output:**
- `final_report.txt` — báo cáo thống kê chi tiết
- `neutral_low_confidence.txt` — 10 câu neutral confidence thấp nhất
- `aspect_sentiment_counts.csv` — bảng `(aspect, pos, neu, neg)`

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q transformers==4.46.0 accelerate==0.34.0 pyarrow pandas tqdm gdown

In [3]:
import os, gc, datetime, glob
import numpy as np
import pandas as pd
import torch
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as _pads   # hỗ trợ cả file đơn lẻ lẫn thư mục parquet
import pyarrow.compute as pc
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
)

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR      = "/content/drive/MyDrive/outputs_electronics_cleaning"
INPUT_PARQUET = f"{BASE_DIR}/cleaned_part1_step2.parquet"
# INPUT_PARQUET có thể là: (1) file .parquet đơn lẻ, hoặc (2) thư mục parquet
# _pads.dataset() xử lý được cả hai trường hợp

# ── Model paths — tải về local /content/ để tránh lỗi Drive timeout ──────────
GATE_MODEL_PATH = "/content/gate_model"
ATE_MODEL_PATH  = "/content/ate_phase2"
ASC_MODEL_PATH  = "/content/asc_phase3"

# ── Output (vẫn lưu trên Drive) ───────────────────────────────────────────────
OUTPUT_DIR         = f"{BASE_DIR}/final_run"
CHUNK_DIR          = f"{OUTPUT_DIR}/chunks"
CHECKPOINT_PARQUET = f"{OUTPUT_DIR}/inference_results.parquet"
REPORT_PATH        = f"{OUTPUT_DIR}/final_report.txt"
NEUTRAL_LOW_PATH   = f"{OUTPUT_DIR}/neutral_low_confidence.txt"
ASPECT_CSV_PATH    = f"{OUTPUT_DIR}/aspect_sentiment_counts.csv"
META_PATH          = f"{OUTPUT_DIR}/meta.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHUNK_DIR,  exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
GATE_THRESHOLD = 0.7
SPAN_THRESHOLD = 0.5
MAX_LENGTH     = 192
MAX_LENGTH_ASC = 128
CHUNK_SIZE     = 50_000
CATEGORY_NAME  = "Electronics_part1"

# ── FP16: ~1.5–2× nhanh hơn, ~50% ít VRAM hơn trên T4 ───────────────────────
USE_FP16 = True

BATCH_SIZE_GATE = 512 if USE_FP16 else 256
BATCH_SIZE_ATE  = 256 if USE_FP16 else 128
BATCH_SIZE_ASC  = 256 if USE_FP16 else 128

# ── Multi-account sharding ─────────────────────────────────────────────────────
NUM_SHARDS = 9   # tổng số account
SHARD_ID   = 7   # ← THAY ĐỔI CHO TỪNG ACCOUNT: 0 → 8

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device             : {device}")
if device == "cuda":
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    print(f"VRAM               : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Gate threshold     : {GATE_THRESHOLD}")
print(f"Span threshold     : {SPAN_THRESHOLD}")
print(f"FP16               : {USE_FP16}")
print(f"Batch Gate/ATE/ASC : {BATCH_SIZE_GATE}/{BATCH_SIZE_ATE}/{BATCH_SIZE_ASC}")
print(f"Chunk size         : {CHUNK_SIZE:,}")
print(f"Shard              : {SHARD_ID} / {NUM_SHARDS}")
print(f"Output dir         : {OUTPUT_DIR}")


Device             : cuda
GPU                : Tesla T4
VRAM               : 15.6 GB
Gate threshold     : 0.7
Span threshold     : 0.5
FP16               : True
Batch Gate/ATE/ASC : 512/256/256
Chunk size         : 50,000
Shard              : 7 / 9
Output dir         : /content/drive/MyDrive/outputs_electronics_cleaning/final_run


In [4]:
import gdown

GATE_FOLDER_ID = "1YCF7tB8Waajw2o2C_d6C4eoOIhjzFio0"
ATE_FOLDER_ID  = "1ERpB4Nxk5wFfCnuAEXihUQXotf12b27Y"
ASC_FOLDER_ID  = "1rV1Jr4uduFvtEd-I7WjPz41vcawi80gv"

# ── Download models về /content/ (nhanh hơn và không bị Drive timeout) ────────
_models = [
    ("Gate", GATE_FOLDER_ID, GATE_MODEL_PATH),
    ("ATE",  ATE_FOLDER_ID,  ATE_MODEL_PATH),
    ("ASC",  ASC_FOLDER_ID,  ASC_MODEL_PATH),
]

for name, folder_id, out_path in _models:
    assert "TODO" not in folder_id, \
        f"Chưa điền Folder ID cho model {name}! Sửa biến {name}_FOLDER_ID ở trên."

    # Kiểm tra đã download chưa → bỏ qua nếu rồi (tránh re-download khi re-run)
    _config = os.path.join(out_path, "config.json")
    if os.path.isfile(_config):
        print(f"  [skip] {name} đã có tại {out_path}")
        continue

    print(f"\nDownloading {name} → {out_path} ...")
    gdown.download_folder(
        id           = folder_id,
        output       = out_path,
        quiet        = False,
        use_cookies  = False,   # False để tránh lỗi permission
        remaining_ok = True,    # bỏ qua file lỗi, tiếp tục các file còn lại
    )
    assert os.path.isfile(_config), \
        f"Download {name} thất bại — không tìm thấy config.json tại {out_path}"
    print(f"  ✓ {name} OK")

print("\nTất cả models sẵn sàng:")
print(f"  Gate : {GATE_MODEL_PATH}")
print(f"  ATE  : {ATE_MODEL_PATH}")
print(f"  ASC  : {ASC_MODEL_PATH}")


  [skip] Gate đã có tại /content/gate_model
  [skip] ATE đã có tại /content/ate_phase2
  [skip] ASC đã có tại /content/asc_phase3

Tất cả models sẵn sàng:
  Gate : /content/gate_model
  ATE  : /content/ate_phase2
  ASC  : /content/asc_phase3


In [5]:
import json

# ── Mở dataset (hỗ trợ cả file đơn lẻ lẫn thư mục partitioned) ──────────────
# _pads.dataset() tự phát hiện: nếu là directory → đọc tất cả *.parquet bên trong
_ds = _pads.dataset(INPUT_PARQUET, format="parquet")

# Đếm số dòng từ metadata (0 MB RAM, không đọc data)
initial_sentence_count = _ds.count_rows()

# Chỉ load cột review_id để đếm unique (~600 MB thay vì 10–12 GB)
_review_tbl          = _ds.to_table(columns=["review_id"])
initial_review_count = pc.count_distinct(_review_tbl.column("review_id")).as_py()
del _review_tbl
gc.collect()

# Lưu meta để dùng lại khi reload từ checkpoint
with open(META_PATH, "w") as f:
    json.dump({
        "initial_sentence_count": initial_sentence_count,
        "initial_review_count"  : initial_review_count,
    }, f)

print(f"Tổng số câu (p2 output) : {initial_sentence_count:>10,}")
print(f"Tổng số review ban đầu  : {initial_review_count:>10,}")

# Xem 3 dòng mẫu (Dataset.head() chỉ đọc đúng 3 dòng, không load cả file)
_COLS   = ["parent_asin", "review_id", "sentence_id", "sentence_text", "rating"]
_sample = _ds.head(3, columns=_COLS).to_pandas()
print(f"\nSample:")
print(_sample.to_string())
del _sample


Tổng số câu (p2 output) : 39,063,570
Tổng số review ban đầu  : 15,397,473

Sample:
  parent_asin  review_id  sentence_id                                                                                                                          sentence_text  rating
0  B075T4MXR9         29            1                                                   fire protection this cover fits like a glove and at the price you just can t beat it     5.0
1  B0774RGDT5       2040            1  the only negative [GENERIC_NOUN] noticed is that the size of the disk reported by both the operating system and the dvd recorder is 7     3.0
2  B07TXY9LLS       2250            1                                                         on the surface pro 4 the skin does not fit all the [GENERIC_NOUN] to the edges     3.0


In [6]:
# ── Load 3 models ──────────────────────────────────────────────────────────────
_dtype = torch.float16 if (USE_FP16 and device == "cuda") else torch.float32
print("=" * 55)
print(f"Model dtype : {_dtype}")

print("\nLoading Gate model ...")
gate_tokenizer = AutoTokenizer.from_pretrained(GATE_MODEL_PATH)
gate_model     = (
    AutoModelForSequenceClassification
    .from_pretrained(GATE_MODEL_PATH, torch_dtype=_dtype)
    .to(device).eval()
)
assert gate_model.config.num_labels == 2, \
    f"Gate: expected 2 labels, got {gate_model.config.num_labels}"
gate_id2label = gate_model.config.id2label
GATE_POS_ID   = next(
    (k for k, v in gate_id2label.items()
     if str(v) in ("1", "LABEL_1", "positive", "has_aspect")),
    1,
)
print(f"  Labels     : {gate_id2label}")
print(f"  POS class  : id={GATE_POS_ID}")

print("\nLoading ATE model ...")
ate_tokenizer = AutoTokenizer.from_pretrained(ATE_MODEL_PATH)
ate_model     = (
    AutoModelForTokenClassification
    .from_pretrained(ATE_MODEL_PATH, torch_dtype=_dtype)
    .to(device).eval()
)
ate_id2label = ate_model.config.id2label
B_ID = next((k for k, v in ate_id2label.items() if str(v).upper().startswith("B")), None)
I_ID = next((k for k, v in ate_id2label.items() if str(v).upper().startswith("I")), None)
assert B_ID is not None and I_ID is not None, \
    f"Không tìm thấy B/I labels: {ate_id2label}"
print(f"  Labels     : {ate_id2label}")
print(f"  B_ID={B_ID}, I_ID={I_ID}")

print("\nLoading ASC model ...")
asc_tokenizer = AutoTokenizer.from_pretrained(ASC_MODEL_PATH)
asc_model     = (
    AutoModelForSequenceClassification
    .from_pretrained(ASC_MODEL_PATH, torch_dtype=_dtype)
    .to(device).eval()
)
assert asc_model.config.num_labels == 3, \
    f"ASC: expected 3 labels, got {asc_model.config.num_labels}"
asc_id2label = asc_model.config.id2label
NEG_ID = next((k for k, v in asc_id2label.items() if "neg" in str(v).lower()), 0)
NEU_ID = next((k for k, v in asc_id2label.items() if "neu" in str(v).lower()), 1)
POS_ID = next((k for k, v in asc_id2label.items() if "pos" in str(v).lower()), 2)
SENT_LABEL = {NEG_ID: "neg", NEU_ID: "neu", POS_ID: "pos"}
print(f"  Labels     : {asc_id2label}")
print(f"  NEG={NEG_ID}, NEU={NEU_ID}, POS={POS_ID}")

print("\n" + "=" * 55)
print("All models loaded OK.")


Model dtype : torch.float16

Loading Gate model ...
  Labels     : {0: 'LABEL_0', 1: 'LABEL_1'}
  POS class  : id=1

Loading ATE model ...
  Labels     : {0: 'O', 1: 'B-ASP', 2: 'I-ASP'}
  B_ID=1, I_ID=2

Loading ASC model ...
  Labels     : {0: 'negative', 1: 'neutral', 2: 'positive'}
  NEG=0, NEU=1, POS=2

All models loaded OK.


In [7]:
# ── BIO decode (vectorized per batch) ─────────────────────────────────────────
def _decode_bio_batch(texts, offsets_np, probs_np, B_ID, I_ID, threshold):
    results = []
    for j in range(len(texts)):
        text      = texts[j]
        sc_arr    = offsets_np[j, :, 0]
        ec_arr    = offsets_np[j, :, 1]
        valid     = sc_arr != ec_arr
        prob_j    = probs_np[j]
        pred_ids  = np.argmax(prob_j, axis=-1)
        pred_conf = prob_j[np.arange(len(pred_ids)), pred_ids]
        is_b = valid & (pred_ids == B_ID) & (pred_conf >= threshold)
        is_i = valid & (pred_ids == I_ID) & (pred_conf >= threshold)
        aspects = []
        span_start = span_end = None
        for ki in np.where(valid)[0]:
            if is_b[ki]:
                if span_start is not None:
                    asp = text[span_start:span_end].strip()
                    if asp: aspects.append(asp)
                span_start, span_end = int(sc_arr[ki]), int(ec_arr[ki])
            elif is_i[ki] and span_start is not None:
                span_end = int(ec_arr[ki])
            else:
                if span_start is not None:
                    asp = text[span_start:span_end].strip()
                    if asp: aspects.append(asp)
                    span_start = span_end = None
        if span_start is not None:
            asp = text[span_start:span_end].strip()
            if asp: aspects.append(asp)
        results.append(aspects)
    return results


def _chunk_path(idx):
    return f"{CHUNK_DIR}/chunk_{idx:06d}.parquet"

def _chunk_done(idx):
    """True nếu chunk đã xử lý (có parquet data hoặc .skip marker)."""
    return os.path.exists(_chunk_path(idx)) or os.path.exists(_chunk_path(idx) + ".skip")


# ── Mở dataset SỚM để tính n_chunks chính xác từ fragment metadata ───────────
# (đọc footer parquet, 0 MB data)
_ds_input = _pads.dataset(INPUT_PARQUET, format="parquet")

# n_chunks tính từ số batch THỰC TẾ của to_batches():
#   mỗi row group → ceil(rg_rows / CHUNK_SIZE) batches
# Dùng cách này để tránh lệch shard range khi row group size ≠ CHUNK_SIZE
_frags   = list(_ds_input.get_fragments())
n_chunks = sum(
    (frag.metadata.row_group(rg).num_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
    for frag in _frags
    for rg in range(frag.metadata.num_row_groups)
)
del _frags

# ── Setup shard range ─────────────────────────────────────────────────────────
n_total     = initial_sentence_count
shard_start = (n_chunks * SHARD_ID)       // NUM_SHARDS
shard_end   = (n_chunks * (SHARD_ID + 1)) // NUM_SHARDS
shard_size  = shard_end - shard_start

# Progress toàn cục (kể cả các shard khác đã chạy trước)
total_written = sum(
    pq.ParquetFile(_chunk_path(i)).metadata.num_rows
    for i in range(n_chunks) if os.path.exists(_chunk_path(i))
)
global_done = sum(1 for i in range(n_chunks) if _chunk_done(i))
shard_done  = sum(1 for i in range(shard_start, shard_end) if _chunk_done(i))

print(f"Dataset total    : {n_total:,} câu  /  {n_chunks} chunks (từ fragment metadata)")
print(f"Shard {SHARD_ID}/{NUM_SHARDS}       : chunk {shard_start}–{shard_end-1}  ({shard_size} chunks)")
print(f"Shard progress   : {shard_done}/{shard_size} done")
print(f"Global progress  : {global_done}/{n_chunks} done  ({total_written:,} rows)\n")

_COLS     = ["parent_asin", "review_id", "sentence_id", "sentence_text", "rating"]
_progress = tqdm(total=shard_size, initial=shard_done, desc=f"Shard {SHARD_ID}")

# ── Inference loop (chỉ xử lý shard của account này) ─────────────────────────
for chunk_idx, _arrow_batch in enumerate(
    _ds_input.to_batches(batch_size=CHUNK_SIZE, columns=_COLS)
):
    if chunk_idx >= shard_end:      # đã qua shard của mình → dừng đọc
        del _arrow_batch
        break

    if chunk_idx < shard_start:     # chưa đến shard của mình → bỏ qua
        del _arrow_batch
        continue

    cp = _chunk_path(chunk_idx)

    if _chunk_done(chunk_idx):      # resume: bỏ qua chunk đã xử lý
        del _arrow_batch
        _progress.update(1)
        continue

    chunk = _arrow_batch.to_pandas()
    chunk["rating"] = chunk["rating"].astype(int)
    del _arrow_batch
    texts = chunk["sentence_text"].tolist()

    # ── 1. Gate ───────────────────────────────────────────────────────────────
    gate_probs_list = []
    with torch.inference_mode():
        for i in range(0, len(texts), BATCH_SIZE_GATE):
            batch  = texts[i : i + BATCH_SIZE_GATE]
            enc    = gate_tokenizer(batch, padding=True, truncation=True,
                                    max_length=MAX_LENGTH, return_tensors="pt")
            logits = gate_model(**{k: v.to(device) for k, v in enc.items()}).logits
            gate_probs_list.extend(
                torch.softmax(logits.float(), dim=-1)[:, GATE_POS_ID].cpu().numpy().tolist()
            )
            del enc, logits

    gate_probs   = np.array(gate_probs_list)
    gate_indices = np.where(gate_probs >= GATE_THRESHOLD)[0]
    del gate_probs, gate_probs_list

    if len(gate_indices) == 0:
        del chunk, texts, gate_indices
        gc.collect()
        if device == "cuda": torch.cuda.empty_cache()
        open(cp + ".skip", "w").close()
        _progress.update(1)
        continue

    # ── 2. ATE ────────────────────────────────────────────────────────────────
    ate_texts            = [texts[i] for i in gate_indices]
    aspects_per_sentence = []

    with torch.inference_mode():
        for i in range(0, len(ate_texts), BATCH_SIZE_ATE):
            batch_t = ate_texts[i : i + BATCH_SIZE_ATE]
            enc     = ate_tokenizer(batch_t, padding=True, truncation=True,
                                    max_length=MAX_LENGTH, return_tensors="pt",
                                    return_offsets_mapping=True)
            offsets = enc.pop("offset_mapping").numpy()
            logits  = ate_model(**{k: v.to(device) for k, v in enc.items()}).logits
            probs   = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            del enc, logits
            aspects_per_sentence.extend(
                _decode_bio_batch(batch_t, offsets, probs, B_ID, I_ID, SPAN_THRESHOLD)
            )
            del probs, offsets

    # ── 3. Build ASC inputs ───────────────────────────────────────────────────
    asc_inputs = []
    asc_meta   = []

    for orig_idx, aspects in zip(gate_indices, aspects_per_sentence):
        if not aspects:
            continue
        row        = chunk.iloc[orig_idx]
        clean_sent = (row["sentence_text"]
                      .replace("[GENERIC_NOUN]", "thing")
                      .replace("[DOMAIN_NOISE]", "item"))
        for asp in aspects:
            clean_asp = (asp.replace("[GENERIC_NOUN]", "thing")
                            .replace("[DOMAIN_NOISE]", "item"))
            pos_in = clean_sent.lower().find(clean_asp.lower())
            if pos_in >= 0:
                ep     = pos_in + len(clean_asp)
                marked = (f"{clean_sent[:pos_in]}"
                          f"[ASP] {clean_sent[pos_in:ep]} [/ASP]"
                          f"{clean_sent[ep:]}")
            else:
                marked = f"[ASP] {clean_asp} [/ASP] {clean_sent}"
            asc_inputs.append(marked)
            asc_meta.append({
                "review_id"    : row["review_id"],
                "sentence_id"  : int(row["sentence_id"]),
                "sentence_text": row["sentence_text"],
                "rating"       : int(row["rating"]),
                "aspect"       : clean_asp,
            })

    del chunk, texts, gate_indices, ate_texts, aspects_per_sentence

    if not asc_inputs:
        del asc_inputs, asc_meta
        gc.collect()
        if device == "cuda": torch.cuda.empty_cache()
        open(cp + ".skip", "w").close()
        _progress.update(1)
        continue

    # ── 4. ASC ────────────────────────────────────────────────────────────────
    sentiments_list  = []
    confidences_list = []

    with torch.inference_mode():
        for i in range(0, len(asc_inputs), BATCH_SIZE_ASC):
            batch  = asc_inputs[i : i + BATCH_SIZE_ASC]
            enc    = asc_tokenizer(batch, padding=True, truncation=True,
                                   max_length=MAX_LENGTH_ASC, return_tensors="pt")
            logits = asc_model(**{k: v.to(device) for k, v in enc.items()}).logits
            p      = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            sentiments_list.extend(np.argmax(p, axis=1).tolist())
            confidences_list.extend(np.max(p, axis=1).tolist())
            del enc, logits, p

    # ── Ghi chunk (atomic: tmp → rename) ──────────────────────────────────────
    chunk_df = pd.DataFrame([
        {**meta, "sentiment": int(s), "confidence": float(c)}
        for meta, s, c in zip(asc_meta, sentiments_list, confidences_list)
    ])
    tmp_cp = cp + ".tmp"
    pq.write_table(pa.Table.from_pandas(chunk_df, preserve_index=False),
                   tmp_cp, compression="gzip")
    os.rename(tmp_cp, cp)
    total_written += len(chunk_df)

    del asc_inputs, asc_meta, sentiments_list, confidences_list, chunk_df
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    _progress.update(1)

_progress.close()
del _ds_input
gc.collect()

global_done_final = sum(1 for i in range(n_chunks) if _chunk_done(i))
print(f"\nShard {SHARD_ID} hoàn tất!")
print(f"Global: {global_done_final}/{n_chunks} chunks done\n")

# ── Hiển thị tiến độ tất cả shards ────────────────────────────────────────────
print("Tiến độ từng shard:")
for sid in range(NUM_SHARDS):
    s0   = (n_chunks * sid)       // NUM_SHARDS
    s1   = (n_chunks * (sid + 1)) // NUM_SHARDS
    done = sum(1 for i in range(s0, s1) if _chunk_done(i))
    pct  = done / (s1 - s0) * 100
    bar  = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
    mark = " ← account này" if sid == SHARD_ID else ""
    print(f"  Shard {sid}: [{bar}] {done:>4}/{s1-s0} ({pct:5.1f}%){mark}")

# ── Merge: chỉ chạy khi TẤT CẢ chunks của mọi shard đã xong ─────────────────
if global_done_final == n_chunks:
    print("\nTất cả shards hoàn tất → Merge ...")
    chunk_files = sorted(glob.glob(f"{CHUNK_DIR}/chunk_*.parquet"))
    pq_writer   = None
    merged_rows = 0
    for cf in tqdm(chunk_files, desc="Merging"):
        t = pq.read_table(cf)
        if len(t) == 0:
            del t; continue
        merged_rows += len(t)
        if pq_writer is None:
            pq_writer = pq.ParquetWriter(CHECKPOINT_PARQUET, t.schema, compression="gzip")
        pq_writer.write_table(t)
        del t
        gc.collect()
    if pq_writer:
        pq_writer.close()
    gc.collect()
    # KHÔNG load results_df ở đây — tránh tràn RAM (~6–8 GB cho 27M rows)
    # Chỉ đọc metadata footer (0 MB) để xác nhận số dòng
    _chk_meta = pq.read_metadata(CHECKPOINT_PARQUET)
    print(f"Merge xong! {_chk_meta.num_rows:,} rows → {CHECKPOINT_PARQUET}")
    del _chk_meta
    print("  → Chạy cell tiếp theo để xem kết quả chi tiết.")
else:
    remaining = n_chunks - global_done_final
    print(f"\nCòn {remaining} chunks chưa xử lý.")
    print(f"Sau khi tất cả {NUM_SHARDS} accounts chạy xong, chạy lại cell này → tự động merge.")



Dataset total    : 39,063,570 câu  /  800 chunks (từ fragment metadata)
Shard 7/9       : chunk 622–710  (89 chunks)
Shard progress   : 89/89 done
Global progress  : 800/800 done  (27,212,397 rows)



Shard 7: 100%|##########| 89/89 [00:00<?, ?it/s]


Shard 7 hoàn tất!
Global: 800/800 chunks done

Tiến độ từng shard:
  Shard 0: [████████████████████]   88/88 (100.0%)
  Shard 1: [████████████████████]   89/89 (100.0%)
  Shard 2: [████████████████████]   89/89 (100.0%)
  Shard 3: [████████████████████]   89/89 (100.0%)
  Shard 4: [████████████████████]   89/89 (100.0%)
  Shard 5: [████████████████████]   89/89 (100.0%)
  Shard 6: [████████████████████]   89/89 (100.0%)
  Shard 7: [████████████████████]   89/89 (100.0%) ← account này
  Shard 8: [████████████████████]   89/89 (100.0%)

Tất cả shards hoàn tất → Merge ...


Merging:   0%|          | 0/800 [00:00<?, ?it/s]

Merge xong! 27,212,397 rows → /content/drive/MyDrive/outputs_electronics_cleaning/final_run/inference_results.parquet
  → Chạy cell tiếp theo để xem kết quả chi tiết.


In [7]:
# ── Kiểm tra nhanh kết quả đã lưu (RAM-safe: không load toàn bộ DF) ──────────
if not os.path.exists(CHECKPOINT_PARQUET):
    print(f"Chưa có checkpoint tại: {CHECKPOINT_PARQUET}")
    print("Hãy chạy xong cell inference (tất cả 9 shard) để có kết quả merge.")
    raise SystemExit("Dừng cell.")

# Metadata footer (0 MB)
_m = pq.read_metadata(CHECKPOINT_PARQUET)
print(f"Checkpoint : {CHECKPOINT_PARQUET}")
print(f"Rows       : {_m.num_rows:,}")
print(f"Columns    : {pq.read_schema(CHECKPOINT_PARQUET).names}")
del _m

# Sample 3 dòng đầu (chỉ đọc đầu file)
_chk_ds = _pads.dataset(CHECKPOINT_PARQUET, format="parquet")
print(f"\nSample 3 dòng đầu:")
print(_chk_ds.head(3).to_pandas().to_string())

# Phân phối sentiment — chỉ đọc 1 cột int (~100 MB thay vì ~5 GB)
_sent_tbl = _chk_ds.to_table(columns=["sentiment"])
print(f"\nPhân phối sentiment (raw id):")
print(_sent_tbl.column("sentiment").to_pandas().value_counts().sort_index())
del _chk_ds, _sent_tbl
gc.collect()



Checkpoint : /content/drive/MyDrive/outputs_electronics_cleaning/final_run/inference_results.parquet
Rows       : 27,212,397
Columns    : ['review_id', 'sentence_id', 'sentence_text', 'rating', 'aspect', 'sentiment', 'confidence']

Sample 3 dòng đầu:
   review_id  sentence_id                                                                                                                          sentence_text  rating           aspect  sentiment  confidence
0         29            1                                                   fire protection this cover fits like a glove and at the price you just can t beat it       5  fire protection          2    0.999967
1         29            1                                                   fire protection this cover fits like a glove and at the price you just can t beat it       5            cover          2    0.999967
2       2040            1  the only negative [GENERIC_NOUN] noticed is that the size of the disk reported by both the oper

11

In [8]:
# ── (Tuỳ chọn) Load từ checkpoint khi kernel restart ─────────────────────────
# Chạy cell này THAY VÌ cell inference nếu đã có checkpoint.
# Yêu cầu: cell 4 (config) và cell 6 (load models / SENT_LABEL) đã chạy.

# import json
# results_df = pd.read_parquet(CHECKPOINT_PARQUET)
# with open(META_PATH) as f:
#     _meta = json.load(f)
# initial_sentence_count = _meta["initial_sentence_count"]
# initial_review_count   = _meta["initial_review_count"]
# print(f"Loaded {len(results_df):,} rows from checkpoint")
# print(f"Initial sentences : {initial_sentence_count:,}")
# print(f"Initial reviews   : {initial_review_count:,}")

In [9]:
# ── Tính toán thống kê ────────────────────────────────────────────────────────

# Fallback cho trường hợp load từ checkpoint (kernel restart)
if "initial_sentence_count" not in dir() or initial_sentence_count == 0:
    import json
    with open(META_PATH) as f:
        _meta = json.load(f)
    initial_sentence_count = _meta["initial_sentence_count"]
    initial_review_count   = _meta["initial_review_count"]
    print(f"[Loaded from meta] sentences={initial_sentence_count:,}  reviews={initial_review_count:,}")

# ══════════════════════════════════════════════════════════════════════════════
# PASS 1: Load KHÔNG có sentence_text (~1.5 GB thay vì ~5–6 GB)
#   → Tính tất cả stats không cần text; tìm 10 neutral low-conf rows
# ══════════════════════════════════════════════════════════════════════════════
_STAT_COLS = ["review_id", "sentence_id", "rating", "aspect", "sentiment", "confidence"]
print("Pass 1: Loading results (không có sentence_text) ...")
results_df = pd.read_parquet(CHECKPOINT_PARQUET, columns=_STAT_COLS)
results_df["sentiment"]  = results_df["sentiment"].astype("int8")
results_df["rating"]     = results_df["rating"].astype("int8")
results_df["confidence"] = results_df["confidence"].astype("float32")
results_df["sent_label"] = pd.Categorical(
    results_df["sentiment"].map(SENT_LABEL),
    categories=["neg", "neu", "pos"],
)
print(f"  {len(results_df):,} rows  "
      f"({results_df.memory_usage(deep=True).sum() / 1e9:.1f} GB)")

# ── [1] Review counts ─────────────────────────────────────────────────────────
aspect_review_count = results_df["review_id"].nunique()
multi_aspect_count  = (
    results_df.groupby("review_id")["aspect"].nunique() > 1
).sum()

# ── [2] Global sentiment distribution ────────────────────────────────────────
total_pairs   = len(results_df)
global_counts = (
    results_df["sent_label"]
    .value_counts()
    .reindex(["neg", "neu", "pos"], fill_value=0)
)

# ── [3] Sentiment theo rating (1–5) ──────────────────────────────────────────
rating_sent = (
    results_df
    .groupby(["rating", "sent_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["neg", "neu", "pos"], fill_value=0)
)
_rt = rating_sent.sum(axis=1)
rating_sent["total"] = _rt
rating_sent["neg_%"] = (rating_sent["neg"] / _rt * 100).round(2)
rating_sent["neu_%"] = (rating_sent["neu"] / _rt * 100).round(2)
rating_sent["pos_%"] = (rating_sent["pos"] / _rt * 100).round(2)

# ── [4–7] Bảng aspect × sentiment ────────────────────────────────────────────
asp_sent = (
    results_df
    .groupby(["aspect", "sent_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["neg", "neu", "pos"], fill_value=0)
)
asp_sent["total"]    = asp_sent.sum(axis=1)
asp_sent["polar"]    = asp_sent["pos"] + asp_sent["neg"]
asp_sent["abs_diff"] = (asp_sent["pos"] - asp_sent["neg"]).abs()

top20_freq     = asp_sent.nlargest(20, "total")[["neg", "neu", "pos", "total"]]
top20_positive = asp_sent.nlargest(20, "pos") [["neg", "neu", "pos", "total"]]
top20_negative = asp_sent.nlargest(20, "neg") [["neg", "neu", "pos", "total"]]
top20_divisive = (
    asp_sent[asp_sent["polar"] >= 10]
    .nsmallest(20, "abs_diff")
    [["neg", "neu", "pos", "polar", "abs_diff"]]
)

# ── [9] ASC confidence statistics ────────────────────────────────────────────
conf_stats = (
    results_df
    .groupby("sent_label")["confidence"]
    .agg(
        mean   = "mean",
        median = "median",
        Q1     = lambda x: x.quantile(0.25),
        Q3     = lambda x: x.quantile(0.75),
    )
    .reindex(["neg", "neu", "pos"], fill_value=0)
    .round(4)
)

# ── [10] Neutral low-conf — lưu metadata hàng (chưa có sentence_text) ────────
_neu_meta = (
    results_df[results_df["sent_label"] == "neu"]
    .nsmallest(10, "confidence")
    [["review_id", "sentence_id", "aspect", "confidence", "rating"]]
    .reset_index(drop=True)
)

# Giải phóng results_df (không cần nữa)
del results_df
gc.collect()
print("Pass 1 xong — results_df đã giải phóng.\n")

# ══════════════════════════════════════════════════════════════════════════════
# PASS 2: Tính char_len qua PyArrow — load sentence_text tạm thời (~2.5 GB)
#   giải phóng ngay sau khi tính xong
# ══════════════════════════════════════════════════════════════════════════════
print("Pass 2: Computing char_len (sentence_text column only) ...")
_chk_ds   = _pads.dataset(CHECKPOINT_PARQUET, format="parquet")
_text_tbl = _chk_ds.to_table(columns=["sentiment", "sentence_text"])

# Tính độ dài trong Arrow (compact binary, không chuyển sang Python strings)
_len_tbl = pa.table({
    "sentiment": _text_tbl.column("sentiment"),
    "char_len" : pc.cast(pc.utf8_length(_text_tbl.column("sentence_text")), pa.int32()),
})
del _text_tbl; gc.collect()

_len_df = _len_tbl.to_pandas()
del _len_tbl; gc.collect()

_len_df["sent_label"] = pd.Categorical(
    _len_df["sentiment"].astype("int8").map(SENT_LABEL),
    categories=["neg", "neu", "pos"],
)
len_stats = (
    _len_df
    .groupby("sent_label")["char_len"]
    .agg(avg_char_len="mean", total_char_len="sum")
    .reindex(["neg", "neu", "pos"], fill_value=0)
    .round(2)
)
del _len_df; gc.collect()
print("Pass 2 xong.\n")

# ══════════════════════════════════════════════════════════════════════════════
# PASS 3: Load sentence_text chỉ cho 10 hàng neutral low-conf (targeted)
#   dùng filter trên review_id → chỉ đọc vài KB data
# ══════════════════════════════════════════════════════════════════════════════
print("Pass 3: Loading sentence_text cho 10 neutral low-conf rows ...")
_rev_ids   = pa.array(_neu_meta["review_id"].tolist())
_texts_df  = (
    _chk_ds
    .to_table(
        filter  = pc.is_in(pc.field("review_id"), _rev_ids),
        columns = ["review_id", "sentence_id", "sentence_text"],
    )
    .to_pandas()
    .drop_duplicates(["review_id", "sentence_id"])
)
neutral_low_10 = (
    _neu_meta
    .merge(_texts_df, on=["review_id", "sentence_id"], how="left")
    .head(10)
    .reset_index(drop=True)
)
del _neu_meta, _texts_df, _chk_ds, _rev_ids
gc.collect()
print("Pass 3 xong.\n")

print("=== Thống kê hoàn tất ===")
print(f"Review ban đầu           : {initial_review_count:>10,}")
print(f"Review có aspect (≥1)    : {aspect_review_count:>10,}")
print(f"Review có >1 aspect      : {multi_aspect_count:>10,}")
print(f"Tổng (câu, aspect) pairs : {total_pairs:>10,}")
print(f"\nGlobal sentiment counts:")
for lbl in ["pos", "neu", "neg"]:
    cnt = int(global_counts[lbl])
    print(f"  {lbl}: {cnt:>8,}  ({cnt / total_pairs * 100:.2f}%)")



Pass 1: Loading results (không có sentence_text) ...
  27,212,397 rows  (2.2 GB)


/tmp/ipykernel_81516/4290013611.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["rating", "sent_label"])
/tmp/ipykernel_81516/4290013611.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["aspect", "sent_label"])
/tmp/ipykernel_81516/4290013611.py:81: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("sent_label")["confidence"]


Pass 1 xong — results_df đã giải phóng.

Pass 2: Computing char_len (sentence_text column only) ...


/tmp/ipykernel_81516/4290013611.py:129: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("sent_label")["char_len"]


Pass 2 xong.

Pass 3: Loading sentence_text cho 10 neutral low-conf rows ...
Pass 3 xong.

=== Thống kê hoàn tất ===
Review ban đầu           : 15,397,473
Review có aspect (≥1)    : 11,468,327
Review có >1 aspect      :  6,063,666
Tổng (câu, aspect) pairs : 27,212,397

Global sentiment counts:
  pos: 16,959,349  (62.32%)
  neu:  400,582  (1.47%)
  neg: 9,852,466  (36.21%)


In [10]:
# ── Tạo báo cáo văn bản ───────────────────────────────────────────────────────
SEP  = "=" * 72
SEP2 = "-" * 72
lines = []

def add(s=""): lines.append(s)

add(SEP)
add(f"FINAL PIPELINE REPORT — {CATEGORY_NAME}")
add(f"Generated      : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
add(f"Gate threshold : {GATE_THRESHOLD}  |  Span threshold : {SPAN_THRESHOLD}  |  ASC : argmax")
add(SEP)

# [1] Dataset
add()
add("[1] DATASET")
add(f"  Tổng số câu (p2 output)           : {initial_sentence_count:>10,}")
add(f"  Tổng số review ban đầu             : {initial_review_count:>10,}")
add(f"  Review có ít nhất 1 aspect         : {aspect_review_count:>10,}")
add(f"  Review có nhiều hơn 1 aspect       : {multi_aspect_count:>10,}")
add(f"  Tổng số (câu, aspect) pairs        : {total_pairs:>10,}")

# [2] Global sentiment
add()
add("[2] PHÂN PHỐI SENTIMENT — TOÀN BỘ")
add(f"  {'Sentiment':<10} {'Count':>10}  {'Tỉ lệ':>8}")
add(f"  {SEP2[:35]}")
for lbl in ["pos", "neu", "neg"]:
    cnt = int(global_counts[lbl])
    add(f"  {lbl:<10} {cnt:>10,}  {cnt / total_pairs * 100:>7.2f}%")

# [3] Per-rating
add()
add("[3] PHÂN PHỐI SENTIMENT THEO RATING (1–5)")
add(rating_sent.to_string())

# [4] Top 20 frequency
add()
add("[4] TOP 20 ASPECT XUẤT HIỆN NHIỀU NHẤT")
add(top20_freq.to_string())

# [5] Top 20 positive
add()
add("[5] TOP 20 ASPECT CÓ NHIỀU POSITIVE NHẤT (theo count)")
add(top20_positive.to_string())

# [6] Top 20 negative
add()
add("[6] TOP 20 ASPECT CÓ NHIỀU NEGATIVE NHẤT (theo count)")
add(top20_negative.to_string())

# [7] Top 20 divisive
add()
add("[7] TOP 20 ASPECT GÂY CHIA RẼ NHẤT  (polar≥10, |pos−neg| nhỏ nhất)")
add(top20_divisive.to_string())

# [8] Sentence length
add()
add("[8] ĐỘ DÀI CÂU THEO SENTIMENT (ký tự)")
add(f"  {'Sentiment':<8}  {'Avg length':>12}  {'Total length':>16}")
add(f"  {SEP2[:42]}")
for lbl in ["pos", "neu", "neg"]:
    r = len_stats.loc[lbl]
    add(f"  {lbl:<8}  {r['avg_char_len']:>12.2f}  {int(r['total_char_len']):>16,}")

# [9] Confidence stats
add()
add("[9] ASC CONFIDENCE STATISTICS THEO SENTIMENT")
add(f"  {'Sentiment':<8}  {'Mean':>8}  {'Median':>8}  {'Q1':>8}  {'Q3':>8}")
add(f"  {SEP2[:50]}")
for lbl in ["pos", "neu", "neg"]:
    r = conf_stats.loc[lbl]
    add(f"  {lbl:<8}  {r['mean']:>8.4f}  {r['median']:>8.4f}  {r['Q1']:>8.4f}  {r['Q3']:>8.4f}")

# [10] Neutral low-confidence
add()
add("[10] 10 CÂU NEUTRAL VỚI CONFIDENCE THẤP NHẤT")
add(SEP2)
for i, row in neutral_low_10.iterrows():
    add(f"  [{i+1}] conf={row['confidence']:.4f} | rating={row['rating']}")
    add(f"      sentence : {row['sentence_text'][:120]}")
    add(f"      aspect   : {row['aspect']}")

add()
add(SEP)
add("OUTPUT FILES")
add(f"  Checkpoint   : {CHECKPOINT_PARQUET}")
add(f"  Report       : {REPORT_PATH}")
add(f"  Neutral low  : {NEUTRAL_LOW_PATH}")
add(f"  Aspect CSV   : {ASPECT_CSV_PATH}")
add(SEP)

report_text = "\n".join(lines)
print(report_text)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"\nReport saved: {REPORT_PATH}")

FINAL PIPELINE REPORT — Electronics_part1
Generated      : 2026-05-23 22:43:43
Gate threshold : 0.7  |  Span threshold : 0.5  |  ASC : argmax

[1] DATASET
  Tổng số câu (p2 output)           : 39,063,570
  Tổng số review ban đầu             : 15,397,473
  Review có ít nhất 1 aspect         : 11,468,327
  Review có nhiều hơn 1 aspect       :  6,063,666
  Tổng số (câu, aspect) pairs        : 27,212,397

[2] PHÂN PHỐI SENTIMENT — TOÀN BỘ
  Sentiment       Count     Tỉ lệ
  -----------------------------------
  pos        16,959,349    62.32%
  neu           400,582     1.47%
  neg         9,852,466    36.21%

[3] PHÂN PHỐI SENTIMENT THEO RATING (1–5)
sent_label      neg     neu       pos     total  neg_%  neu_%  pos_%
rating                                                              
1           2386121   36370    501459   2923950  81.61   1.24  17.15
2           1400515   44416    544588   1989519  70.39   2.23  27.37
3           1637057   89397   1052906   2779360  58.90   3.22  37.88

In [11]:
# ── Export output files ───────────────────────────────────────────────────────

# (A) neutral_low_confidence.txt — 10 câu neutral confidence thấp nhất
with open(NEUTRAL_LOW_PATH, "w", encoding="utf-8") as f:
    f.write("10 Câu Neutral — Confidence Thấp Nhất (Error Analysis)\n")
    f.write("=" * 72 + "\n\n")
    for i, row in neutral_low_10.iterrows():
        f.write(f"[{i+1}] confidence = {row['confidence']:.4f}  |  rating = {row['rating']}\n")
        f.write(f"    sentence : {row['sentence_text']}\n")
        f.write(f"    aspect   : {row['aspect']}\n\n")

print(f"File saved: {NEUTRAL_LOW_PATH}")
print(neutral_low_10[["sentence_text", "aspect", "confidence"]].to_string())

# (B) aspect_sentiment_counts.csv — (aspect, pos, neu, neg)
aspect_csv = (
    asp_sent[["pos", "neu", "neg"]]
    .reset_index()
    [["aspect", "pos", "neu", "neg"]]
    .sort_values("aspect")
)
aspect_csv.to_csv(ASPECT_CSV_PATH, index=False, encoding="utf-8")

print(f"\nFile saved: {ASPECT_CSV_PATH}")
print(f"Unique aspects : {len(aspect_csv):,}")
print("\nTop 10 theo pos count:")
print(aspect_csv.nlargest(10, "pos").to_string(index=False))

File saved: /content/drive/MyDrive/outputs_electronics_cleaning/final_run/neutral_low_confidence.txt
                                                                                                                                                                                                   sentence_text        aspect  confidence
0                                                                                       the base plate should either be heavier or deeper to provide better tipping prevention if not mounting to your table top    base plate    0.334275
1                                                                                                                 [DOMAIN_NOISE] featured electronics should last longer than 18 [GENERIC_NOUN] 2 [GENERIC_NOUN]   electronics    0.340408
2                                                                                                                                                           it would have been nice if the height 